# 🔍 Support Integrity Auditor (SIA)
## MARS Open Projects 2026 — AI/ML Problem Statement 1

> **Full Reproducible Pipeline**: Pseudo-Label Generation → Classifier Training → Evaluation → Evidence Dossiers

| Stage | Description |
|-------|-------------|
| 1 | Dataset Loading & EDA |
| 2 | Pseudo-Label Generation (3 Signals + Fusion) |
| 3 | Mismatch Label Creation |
| 4 | Baseline Model (TF-IDF + Logistic Regression) |
| 5 | Advanced Model (DeBERTa-v3-small) |
| 6 | Evaluation & Metrics |
| 7 | Evidence Dossier Generation |
| 8 | Adversarial Testing |
| 9 | Ablation Study |

## ⚙️ Setup & Installation

In [ ]:
# Install all dependencies
!pip install -q pandas numpy scikit-learn sentence-transformers transformers \
    torch nltk seaborn matplotlib plotly tqdm joblib

## 📦 Imports

In [ ]:
import sys, json, warnings, re
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

plt.rcParams.update({'figure.facecolor': 'white', 'font.size': 11})
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('✓ All imports successful')

## ⚙️ Configuration

In [ ]:
# ── Priority levels & mappings ──────────────────────────
PRIORITY_LEVELS = ['Low', 'Medium', 'High', 'Critical']
PRIORITY_MAP    = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
PRIORITY_MAP_INV = {v: k for k, v in PRIORITY_MAP.items()}

# ── Signal fusion weights (must sum to 1.0) ──────────────
SIGNAL_WEIGHTS = {'semantic': 0.45, 'resolution_time': 0.25, 'rule_based': 0.30}

# ── Column names ──────────────────────────────────────────
COL_ID       = 'Ticket ID'
COL_SUBJECT  = 'Ticket Subject'
COL_DESC     = 'Ticket Description'
COL_PRIORITY = 'Ticket Priority'
COL_CHANNEL  = 'Ticket Channel'
COL_RT       = 'Resolution Time'
COL_TYPE     = 'Ticket Type'

# ── Keyword lexicons ──────────────────────────────────────
CRITICAL_KW = [
    'server down','payment failed','payment gateway','security breach',
    'production outage','data loss','data breach','system failure',
    'complete outage','cannot access','database down','critical failure',
    'service unavailable','emergency','revenue loss','sla breach',
    'ransomware','intrusion','credentials compromised','all users affected'
]
HIGH_KW = [
    'not working','broken','error','bug','crash','slow','performance',
    'degraded','login issue','cannot login','api error','timeout','500 error'
]
MEDIUM_KW = [
    'help','issue','problem','question','unable to','unexpected behavior'
]
LOW_KW = [
    'feature request','suggestion','feedback','profile','settings',
    'change password','notification','profile picture','username change'
]
ESCALATION_KW = [
    'escalate','manager','legal action','lawsuit','unacceptable',
    'not the first time','repeatedly','still not fixed'
]
NEGATION_WORDS = ['not','never','no','cannot',"can't","won't","doesn't","don't"]

print('✓ Configuration loaded')

## 📊 Stage 1: Dataset Loading & EDA

In [ ]:
TEMPLATES = [
    ('Production payment gateway down',
     'Our production payment gateway is completely down. Customers cannot complete purchases. Revenue impact severe.',
     'Low', 1.5),
    ('Server outage affecting all users',
     'The main application server has crashed. All users are locked out. Immediate action required.',
     'Medium', 2.0),
    ('Security breach detected',
     'Unauthorized access to customer credentials detected. Data breach in progress.',
     'Low', 3.0),
    ('Database corruption',
     'Primary database is corrupted. All data access failing. Backups are 24h old.',
     'Medium', 4.0),
    ('Change profile picture',
     'I want to update my profile photo on the account page.',
     'Critical', 120.0),
    ('Update notification preferences',
     'Can you help me turn off email notifications?',
     'High', 96.0),
    ('Feature request: dark mode',
     'It would be nice to have a dark mode option in the UI.',
     'Critical', 150.0),
    ('Password reset email not received',
     'Requested password reset 10 minutes ago but no email received.',
     'Medium', 8.0),
    ('API returning 500 errors',
     'Integration getting 500 errors on 30% of requests since this morning.',
     'Low', 6.0),
    ('Login page not loading',
     'Cannot access login page at all. Entire team is blocked.',
     'Low', 3.5),
    ('Minor UI alignment issue',
     'The submit button is slightly off-center on mobile devices.',
     'High', 200.0),
    ('Billing inquiry',
     'I have a question about my invoice from last month.',
     'Critical', 72.0),
    ('Data loss after update',
     'After latest update, all saved configurations are gone. Production system.',
     'Low', 5.0),
    ('Slow dashboard loading',
     'Analytics dashboard takes 15 seconds to load. Used to be instant.',
     'Medium', 24.0),
    ('Complete service outage',
     'All services down for enterprise account. Entire company affected.',
     'Medium', 1.0),
    ('SLA breach notification',
     'Breaching SLA with key clients due to ongoing outage. Legal escalation imminent.',
     'Low', 2.0),
    ('Font size preference',
     'Could the default font size be increased slightly?',
     'Critical', 168.0),
    ('OAuth integration broken',
     'OAuth login returning 401 for all users. Nobody can sign in via SSO.',
     'Medium', 4.5),
    ('Ransomware detected',
     'Ransomware has encrypted our backup server. Customer data at risk.',
     'Low', 1.0),
    ('How do I export data',
     'Not sure how to export data to CSV format.',
     'Medium', 36.0),
    ('Transaction failures 100%',
     '100% of payment transactions are failing. No orders can be processed.',
     'Low', 1.5),
    ('Cosmetic bug on profile page',
     'Small spacing issue between avatar and username.',
     'High', 180.0),
    ('SSL certificate expired',
     'SSL certificate expired. All HTTPS requests failing. Browser warnings everywhere.',
     'Medium', 1.0),
    ('Change account timezone',
     'I would like to change my account timezone setting.',
     'Critical', 48.0),
    ('Intrusion detection alert',
     'Multiple failed login attempts from foreign IPs. Possible brute force attack.',
     'Low', 2.0),
]

CHANNELS = ['Email','Chat','Phone','Social Media','Portal']
TYPES    = ['Technical Issue','Billing','Feature Request','Account Management','Bug Report']
PRODUCTS = ['ProductA','ProductB','SaaS Platform','Enterprise Suite','Mobile App']

def generate_dataset(n=2000, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        subj, desc, pri, rt = TEMPLATES[i % len(TEMPLATES)]
        rt_noisy = max(0.5, rt * rng.normal(1.0, 0.15))
        rows.append({
            COL_ID: f'TKT-{i+1:05d}',
            COL_SUBJECT: subj, COL_DESC: desc,
            COL_PRIORITY: pri, COL_CHANNEL: rng.choice(CHANNELS),
            COL_RT: round(rt_noisy, 2), COL_TYPE: rng.choice(TYPES),
            'Customer Email': f'user{i+1}@example.com',
            'Product Purchased': rng.choice(PRODUCTS),
        })
    df = pd.DataFrame(rows)
    df['combined_text'] = (df[COL_SUBJECT].str.lower() + ' ' + df[COL_DESC].str.lower()).str.strip()
    return df

df = generate_dataset(2000)
print(f'Dataset shape: {df.shape}')
df.head(3)

### 📈 EDA Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Support Ticket Dataset — EDA', fontsize=16, fontweight='bold')
PALETTE = {'Low':'#28a745','Medium':'#ffc107','High':'#fd7e14','Critical':'#dc3545'}

# 1. Priority distribution
pri_counts = df[COL_PRIORITY].value_counts().reindex(PRIORITY_LEVELS)
axes[0,0].bar(pri_counts.index, pri_counts.values,
              color=[PALETTE[p] for p in pri_counts.index], edgecolor='white')
axes[0,0].set_title('Priority Distribution'); axes[0,0].set_ylabel('Count')
for i,(k,v) in enumerate(pri_counts.items()): axes[0,0].text(i,v+5,str(v),ha='center',fontweight='bold')

# 2. Resolution time histogram
axes[0,1].hist(df[COL_RT][df[COL_RT]<=200], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0,1].axvline(df[COL_RT].median(), color='red', linestyle='--', label=f'Median={df[COL_RT].median():.0f}h')
axes[0,1].set_title('Resolution Time Distribution'); axes[0,1].set_xlabel('Hours'); axes[0,1].legend()

# 3. RT by priority (boxplot)
data_bp = [df[df[COL_PRIORITY]==p][COL_RT].values for p in PRIORITY_LEVELS]
bp = axes[0,2].boxplot(data_bp, labels=PRIORITY_LEVELS, patch_artist=True)
for patch,p in zip(bp['boxes'],PRIORITY_LEVELS): patch.set_facecolor(PALETTE[p])
axes[0,2].set_title('Resolution Time by Priority'); axes[0,2].set_ylabel('Hours'); axes[0,2].set_ylim(0,200)

# 4. Channel distribution
ch = df[COL_CHANNEL].value_counts()
axes[1,0].bar(ch.index, ch.values, color='#9b59b6', edgecolor='white')
axes[1,0].set_title('Tickets by Channel'); axes[1,0].set_ylabel('Count')
plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# 5. Ticket type distribution
tt = df[COL_TYPE].value_counts()
axes[1,1].barh(tt.index, tt.values, color='#1abc9c', edgecolor='white')
axes[1,1].set_title('Tickets by Type'); axes[1,1].set_xlabel('Count')

# 6. Description length by priority
df['desc_len'] = df[COL_DESC].str.len()
for p in PRIORITY_LEVELS:
    axes[1,2].hist(df[df[COL_PRIORITY]==p]['desc_len'], bins=30, alpha=0.6, label=p, color=PALETTE[p])
axes[1,2].set_title('Description Length by Priority'); axes[1,2].set_xlabel('Chars'); axes[1,2].legend()

plt.tight_layout()
plt.show()

print('\n📊 KEY INSIGHTS:')
print(f'  Priority distribution: {df[COL_PRIORITY].value_counts().to_dict()}')
print(f'  Avg RT (Critical): {df[df[COL_PRIORITY]=="Critical"][COL_RT].mean():.1f}h')
print(f'  Avg RT (Low):      {df[df[COL_PRIORITY]=="Low"][COL_RT].mean():.1f}h')
print(f'  Missing values:    {df.isnull().sum().sum()}')

## 🔬 Stage 2: Pseudo-Label Generation
### Signal C — Rule-Based NLP Scorer

In [ ]:
def count_kw_hits(text, keywords):
    """Negation-aware keyword counter."""
    score = 0.0
    text_lower = text.lower()
    for kw in keywords:
        pattern = re.compile(r'\b' + re.escape(kw) + r'\b')
        for match in pattern.finditer(text_lower):
            window = text_lower[max(0, match.start()-40): match.start()]
            negated = any(re.search(r'\b'+n+r'\b', window) for n in NEGATION_WORDS)
            score += 0.2 if negated else 1.0
    return score

BUSINESS_PATTERNS = [
    r'revenue (loss|impact)', r'customers? (cannot|unable to)',
    r'all users? (affected|blocked)', r'entire (company|team)',
    r'sla (breach|violation)', r'financial impact', r'\b100%\b'
]

def rule_score_single(text):
    """Score one ticket text with rule-based features."""
    c = text.lower()
    feat = {
        'critical_hits': count_kw_hits(c, CRITICAL_KW),
        'high_hits':     count_kw_hits(c, HIGH_KW),
        'medium_hits':   count_kw_hits(c, MEDIUM_KW),
        'low_hits':      count_kw_hits(c, LOW_KW),
        'escalation':    sum(1 for p in ESCALATION_KW if p in c),
        'biz_impact':    sum(1 for p in BUSINESS_PATTERNS if re.search(p, c)),
        'caps_words':    len(re.findall(r'\b[A-Z]{3,}\b', text)),
        'exclamations':  text.count('!'),
    }
    scores = {
        'Critical': feat['critical_hits']*3 + feat['biz_impact']*2 + feat['escalation']*1.5 + feat['caps_words']*0.3,
        'High':     feat['high_hits']*2    + feat['escalation']   + feat['caps_words']*0.4,
        'Medium':   feat['medium_hits']*1.5,
        'Low':      feat['low_hits']*2 + (1.5 if feat['critical_hits']==0 and feat['high_hits']==0 else 0),
    }
    total = sum(scores.values()) + 1e-9
    best  = max(scores, key=scores.get)
    conf  = min(0.95, 0.50 + (scores[best]-sorted(scores.values())[-2]) / (total+1e-9) * 2)
    return best, round(max(0.4, conf), 4), feat

# Apply to dataset
rule_results = [rule_score_single(t) for t in df['combined_text']]
df['rule_severity'] = [r[0] for r in rule_results]
df['rule_score']    = [r[1] for r in rule_results]

print('Rule-Based Severity Distribution:')
print(df['rule_severity'].value_counts().to_string())

### Signal B — Resolution Time Scorer

In [ ]:
import numpy as np

log_rt = np.log1p(df[COL_RT].values)
q15 = np.percentile(log_rt, 15)   # Critical threshold
q35 = np.percentile(log_rt, 35)   # High threshold
q65 = np.percentile(log_rt, 65)   # Medium threshold

print(f'Thresholds → Critical: ≤{np.expm1(q15):.1f}h | High: ≤{np.expm1(q35):.1f}h | Medium: ≤{np.expm1(q65):.1f}h')

def rt_classify(rt_val):
    lv = np.log1p(max(rt_val, 0.1))
    if lv <= q15:  return 'Critical', min(0.95, 0.70 + (q15-lv)/(q15+1e-9)*0.25)
    if lv <= q35:  return 'High',     min(0.90, 0.60 + (q35-lv)/(q35-q15+1e-9)*0.30)
    if lv <= q65:  return 'Medium',   min(0.85, 0.55 + (q65-lv)/(q65-q35+1e-9)*0.25)
    return 'Low', min(0.85, 0.55 + (lv-q65)/(lv+1e-9)*0.20)

rt_results = [rt_classify(v) for v in df[COL_RT].values]
df['rt_severity'] = [r[0] for r in rt_results]
df['rt_score']    = [r[1] for r in rt_results]

print('RT Severity Distribution:')
print(df['rt_severity'].value_counts().to_string())

### Signal A — Semantic Embedding Scorer

In [ ]:
ANCHORS = {
    'Low': [
        'I want to change my profile picture.',
        'Please update my notification preferences.',
        'I have a minor suggestion for the user interface.',
        'Can you help me find the documentation?',
        'I would like to update my account settings.',
    ],
    'Medium': [
        'I cannot access a specific feature in the application.',
        'The report is generating incorrect data for some entries.',
        'I am experiencing intermittent login issues.',
        'The API is returning unexpected results occasionally.',
    ],
    'High': [
        'The application is crashing repeatedly for multiple users.',
        'Authentication is broken and users cannot log in.',
        'Performance is severely degraded and the service is very slow.',
        'A major feature is completely broken affecting our workflow.',
    ],
    'Critical': [
        'Our production server is completely down and no one can access the system.',
        'The payment gateway has failed and customers cannot complete purchases.',
        'We have detected a security breach and customer data may be compromised.',
        'Complete service outage affecting all users with severe revenue impact.',
    ],
}

try:
    from sentence_transformers import SentenceTransformer
    print('Loading all-MiniLM-L6-v2 ...')
    sbert = SentenceTransformer('all-MiniLM-L6-v2')

    anchor_embs = {}
    for level, sents in ANCHORS.items():
        embs = sbert.encode(sents, convert_to_numpy=True, show_progress_bar=False)
        anchor_embs[level] = embs.mean(axis=0)

    def cosine_sim(a, b):
        return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)+1e-9))

    print('Encoding tickets (this may take a moment)...')
    ticket_embs = sbert.encode(df['combined_text'].tolist(),
                                convert_to_numpy=True, show_progress_bar=True, batch_size=128)

    sem_sev, sem_score = [], []
    for emb in ticket_embs:
        sims = {l: cosine_sim(emb, ae) for l, ae in anchor_embs.items()}
        best = max(sims, key=sims.get)
        vals = np.array(list(sims.values()))
        exp_v = np.exp(vals*10)
        conf  = float(exp_v.max()/exp_v.sum())
        sem_sev.append(best); sem_score.append(round(conf, 4))

    df['semantic_severity'] = sem_sev
    df['semantic_score']    = sem_score
    print('Semantic Severity Distribution:')
    print(df['semantic_severity'].value_counts().to_string())

except ImportError:
    print('sentence-transformers not installed — using rule_severity as fallback for Signal A')
    df['semantic_severity'] = df['rule_severity']
    df['semantic_score']    = df['rule_score'] * 0.9

### 🔀 Signal Fusion

In [ ]:
def fuse_signals(sem_sev, sem_sc, rt_sev, rt_sc, rule_sev, rule_sc):
    votes = {'Low':0.0,'Medium':0.0,'High':0.0,'Critical':0.0}
    votes[sem_sev]  += SIGNAL_WEIGHTS['semantic']        * sem_sc
    votes[rt_sev]   += SIGNAL_WEIGHTS['resolution_time'] * rt_sc
    votes[rule_sev] += SIGNAL_WEIGHTS['rule_based']      * rule_sc
    total   = sum(votes.values()) + 1e-9
    winner  = max(votes, key=votes.get)
    conf    = round(votes[winner]/total, 4)
    signals = [sem_sev, rt_sev, rule_sev]
    agree   = round(sum(1 for s in signals if s==winner)/3, 4)
    return winner, conf, agree

def create_mismatch_label(assigned, inferred):
    a, i = PRIORITY_MAP.get(assigned,1), PRIORITY_MAP.get(inferred,1)
    delta = i - a
    if delta == 0: return 'Consistent', '', 0
    return 'Mismatch', ('Hidden Crisis' if delta>0 else 'False Alarm'), delta

fused = [
    fuse_signals(
        row['semantic_severity'], row['semantic_score'],
        row['rt_severity'],       row['rt_score'],
        row['rule_severity'],     row['rule_score'],
    )
    for _, row in df.iterrows()
]

df['inferred_severity'] = [f[0] for f in fused]
df['fusion_confidence'] = [f[1] for f in fused]
df['signal_agreement']  = [f[2] for f in fused]

mm = [create_mismatch_label(row[COL_PRIORITY], row['inferred_severity']) for _,row in df.iterrows()]
df['mismatch_label'] = [m[0] for m in mm]
df['mismatch_type']  = [m[1] for m in mm]
df['severity_delta'] = [m[2] for m in mm]
df['label']          = (df['mismatch_label']=='Mismatch').astype(int)

print(f'Mismatch rate:   {df["label"].mean()*100:.1f}%')
print(f'Hidden Crisis:   {(df["mismatch_type"]=="Hidden Crisis").sum()}')
print(f'False Alarm:     {(df["mismatch_type"]=="False Alarm").sum()}')
print(f'\nInferred Severity Distribution:')
print(df['inferred_severity'].value_counts().to_string())

## 📊 Stage 3: Ablation Study — Signal Agreement

In [ ]:
pairs = [
    ('semantic_severity', 'rt_severity',       'Semantic vs RT'),
    ('semantic_severity', 'rule_severity',     'Semantic vs Rule'),
    ('rt_severity',       'rule_severity',     'RT vs Rule'),
    ('semantic_severity', 'inferred_severity', 'Semantic vs Fused'),
    ('rt_severity',       'inferred_severity', 'RT vs Fused'),
    ('rule_severity',     'inferred_severity', 'Rule vs Fused'),
]
ablation = pd.DataFrame([
    {'Signal Pair': label, 'Agreement Rate': round((df[a]==df[b]).mean(),4)}
    for a,b,label in pairs
])
print('Ablation Study — Pairwise Signal Agreement:')
print(ablation.to_string(index=False))

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(ablation['Signal Pair'], ablation['Agreement Rate'],
               color='#3498db', edgecolor='white')
ax.axvline(0.7, color='red', linestyle='--', label='0.70 baseline')
ax.set_xlabel('Agreement Rate'); ax.set_title('Signal Pairwise Agreement', fontweight='bold')
ax.set_xlim(0,1); ax.legend()
for bar, val in zip(bars, ablation['Agreement Rate']):
    ax.text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center')
plt.tight_layout(); plt.show()

## 🤖 Stage 4: Baseline Model — TF-IDF + Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df, df['label'], test_size=0.2, random_state=RANDOM_SEED, stratify=df['label']
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train mismatch rate: {y_train.mean():.3f}')
print(f'Test  mismatch rate: {y_test.mean():.3f}')

In [ ]:
# ── Feature Engineering ──────────────────────────────────────
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,3),
                         sublinear_tf=True, min_df=2)
ch_enc = LabelEncoder()
tt_enc = LabelEncoder()
scaler = StandardScaler()

def build_features(X, fit=False):
    if fit:
        tfidf_f = tfidf.fit_transform(X['combined_text'])
        ch = ch_enc.fit_transform(X[COL_CHANNEL].fillna('Unknown')).reshape(-1,1)
        tt = tt_enc.fit_transform(X[COL_TYPE].fillna('Unknown')).reshape(-1,1)
    else:
        tfidf_f = tfidf.transform(X['combined_text'])
        ch_safe = X[COL_CHANNEL].fillna('Unknown').map(
            lambda x: x if x in ch_enc.classes_ else ch_enc.classes_[0])
        tt_safe = X[COL_TYPE].fillna('Unknown').map(
            lambda x: x if x in tt_enc.classes_ else tt_enc.classes_[0])
        ch = ch_enc.transform(ch_safe).reshape(-1,1)
        tt = tt_enc.transform(tt_safe).reshape(-1,1)
    rt  = np.log1p(X[COL_RT].fillna(24).astype(float)).values.reshape(-1,1)
    txl = (X['combined_text'].str.len() / 1000.0).values.reshape(-1,1)
    meta = np.hstack([ch, tt, rt, txl])
    if fit: meta = scaler.fit_transform(meta)
    else:   meta = scaler.transform(meta)
    return hstack([tfidf_f, csr_matrix(meta)])

X_tr = build_features(X_train, fit=True)
X_te = build_features(X_test,  fit=False)

lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                         random_state=RANDOM_SEED, solver='lbfgs')
lr.fit(X_tr, y_train)
print(f'TF-IDF vocab size: {len(tfidf.vocabulary_)}')
print('Baseline model trained ✓')

## 📏 Stage 5: Evaluation

In [ ]:
y_pred  = lr.predict(X_te)
y_proba = lr.predict_proba(X_te)

acc      = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
roc      = roc_auc_score(y_test, y_proba[:,1])

from sklearn.metrics import precision_score, recall_score
per_prec = precision_score(y_test, y_pred, average=None)
per_rec  = recall_score(y_test, y_pred, average=None)
per_f1   = f1_score(y_test, y_pred, average=None)

print('=' * 55)
print('  EVALUATION REPORT — Baseline TF-IDF + LR')
print('=' * 55)
print(f'  Accuracy  : {acc*100:.2f}%  (target ≥83%)   {"✓" if acc>=0.83 else "✗"}')
print(f'  Macro F1  : {macro_f1:.4f}    (target ≥0.82)  {"✓" if macro_f1>=0.82 else "✗"}')
print(f'  ROC-AUC   : {roc:.4f}')
print(f'  Consistent: P={per_prec[0]:.3f} R={per_rec[0]:.3f} F1={per_f1[0]:.3f}  {"✓" if per_rec[0]>=0.78 else "✗"}')
print(f'  Mismatch  : P={per_prec[1]:.3f} R={per_rec[1]:.3f} F1={per_f1[1]:.3f}  {"✓" if per_rec[1]>=0.78 else "✗"}')
print('=' * 55)
overall = acc>=0.83 and macro_f1>=0.82 and per_rec[0]>=0.78 and per_rec[1]>=0.78
print(f'  Status: {"✓ ALL THRESHOLDS MET" if overall else "⚠ BELOW THRESHOLD"}')
print('=' * 55)

In [ ]:
# ── Confusion Matrix ────────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(14,5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Consistent','Mismatch'],
            yticklabels=['Consistent','Mismatch'], ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix — Baseline', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_proba[:,1])
axes[1].plot(fpr, tpr, color='#3498db', linewidth=2, label=f'ROC (AUC={roc:.3f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#3498db')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Baseline', fontweight='bold'); axes[1].legend()

plt.tight_layout(); plt.show()

## 🚀 Stage 6: Advanced Model — DeBERTa-v3-small

> Fine-tuning DeBERTa-v3-small on pseudo-labeled data.
> Requires GPU for practical training. Shown here as reference — use `train_pipeline.py` for full training.

In [ ]:
# ── DeBERTa Training Code (run train_pipeline.py for full execution) ──
DEBERTA_AVAILABLE = False
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    DEBERTA_AVAILABLE = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'PyTorch available. Device: {device}')
    print('DeBERTa training → see src/ml/advanced_model.py and train_pipeline.py')
    print('Run: python train_pipeline.py  (GPU recommended)')
except ImportError:
    print('torch/transformers not installed in this environment.')
    print('Run: pip install torch transformers, then: python train_pipeline.py')

# DeBERTa input format used:
example = '[CHANNEL: Email] [TYPE: Technical Issue] [RT: RESOLVED-FAST] '\
          'production server down [SEP] all users locked out complete outage'
print(f'\nExample DeBERTa input:\n  {example}')

## 📋 Stage 7: Evidence Dossier Generation

In [ ]:
def generate_dossier(row):
    """Generates grounded evidence dossier. Zero hallucination guaranteed."""
    if row['mismatch_label'] != 'Mismatch':
        return None

    text = (str(row.get(COL_SUBJECT,'')) + ' ' + str(row.get(COL_DESC,''))).lower()
    evidence = []

    # ── Keyword evidence (verified against ticket text) ──
    for kw in CRITICAL_KW + HIGH_KW:
        if kw.lower() in text:
            src = COL_SUBJECT if kw.lower() in row.get(COL_SUBJECT,'').lower() else COL_DESC
            evidence.append({
                'signal': 'keyword',
                'value': kw,
                'weight': 'Critical severity indicator' if kw in CRITICAL_KW else 'High severity indicator',
                'source_field': src
            })
        if len(evidence) >= 3: break

    # ── Resolution time evidence (actual value from ticket) ──
    rt = float(row.get(COL_RT, 24))
    if rt <= 4:   rt_interp = f'Resolved in {rt:.1f}h — extremely fast, production-critical handling.'
    elif rt <= 24: rt_interp = f'Resolved in {rt:.1f}h — fast, operationally urgent.'
    elif rt <= 72: rt_interp = f'Resolved in {rt:.1f}h — moderate urgency.'
    else:          rt_interp = f'Resolved in {rt:.1f}h — slow, low operational urgency.'
    evidence.append({'signal':'resolution_time','value':f'{rt:.1f} hours',
                     'interpretation': rt_interp, 'source_field': COL_RT})

    # ── Semantic evidence ──
    sem_sc = float(row.get('semantic_score', 0.5))
    evidence.append({'signal':'semantic_embedding',
                     'value': f"Semantic similarity to '{row['inferred_severity']}' anchor: {sem_sc:.3f}",
                     'source_field': f'{COL_SUBJECT} + {COL_DESC}'})

    # ── Constraint analysis (grounded — no hallucination) ──
    assigned = row[COL_PRIORITY]; inferred = row['inferred_severity']
    delta    = abs(int(row['severity_delta']))
    mtype    = row['mismatch_type']
    subj     = str(row.get(COL_SUBJECT,''))[:60]
    ttype    = str(row.get(COL_TYPE,'Unknown'))
    if mtype == 'Hidden Crisis':
        analysis = (f"Ticket '{subj}...' assigned {assigned} but signals converge on "
                    f"{inferred} — gap of {delta} level(s). "
                    f"Resolution time {rt:.1f}h indicates operational urgency. "
                    f"Under-prioritization risks SLA breach for {ttype.lower()} requests.")
    else:
        analysis = (f"Ticket '{subj}...' assigned {assigned} but actual severity is "
                    f"{inferred} — over-prioritized by {delta} level(s). "
                    f"Resolution time {rt:.1f}h is inconsistent with {assigned}-tier urgency. "
                    f"Over-prioritization drains high-urgency agent capacity.")

    conf = float(row.get('fusion_confidence', 0.7))
    conf_label = f'High ({conf:.1%})' if conf>=0.85 else f'Medium ({conf:.1%})' if conf>=0.60 else f'Low ({conf:.1%})'

    return {
        'ticket_id':         str(row.get(COL_ID,'?')),
        'assigned_priority': assigned,
        'inferred_severity': inferred,
        'mismatch_type':     mtype,
        'severity_delta':    int(row['severity_delta']),
        'feature_evidence':  evidence[:5],
        'constraint_analysis': analysis,
        'confidence':        conf_label,
    }

# Generate dossiers for test set predictions
X_test_preds = X_test.copy()
X_test_preds['mismatch_label'] = np.where(y_pred==1,'Mismatch','Consistent')
dossiers = [generate_dossier(row) for _,row in X_test_preds.iterrows()]
dossiers = [d for d in dossiers if d is not None]

print(f'Dossiers generated: {len(dossiers)}')
print('\nSample Dossier:')
print(json.dumps(dossiers[0], indent=2) if dossiers else 'No mismatches in test set')

## ⚔️ Stage 8: Adversarial Testing

In [ ]:
ADVERSARIAL = [
    ('Sarcastic outage',
     'Oh nothing big, just our entire payment system decided to take a nap. Customers cannot buy anything.',
     'Low', True, 'Hidden Crisis'),
    ('Sarcastic minor issue',
     'THIS IS THE WORST THING EVER. The font on the login button is 1px too small.',
     'Critical', True, 'False Alarm'),
    ('Negation trap critical',
     'This is not a server down issue. The system is just completely unusable for all 10,000 enterprise users.',
     'Low', True, 'Hidden Crisis'),
    ('Urgent buried in casual',
     'Hey! Hope you are having a great day. Quick question — our entire database got corrupted.',
     'Low', True, 'Hidden Crisis'),
    ('Formal for trivial',
     'I formally request with utmost urgency that the background color be changed from #1a1a2e to #1a1a2f.',
     'Critical', True, 'False Alarm'),
    ('Indirect business impact',
     'Since this morning our conversion rate has dropped to zero. Finance has called twice.',
     'Low', True, 'Hidden Crisis'),
    ('Keyword abuse sarcasm',
     'SERVER DOWN ERROR CRITICAL EMERGENCY URGENT — just kidding, I need help resetting my password.',
     'Critical', True, 'False Alarm'),
    ('Correct critical',
     'Production database is down. All users locked out. Revenue impact confirmed.',
     'Critical', False, ''),
    ('Correct low',
     'I would like to update my billing address for future invoices.',
     'Low', False, ''),
    ('Security breach understated',
     'We noticed some unusual account activity. About 50,000 user records accessed by unknown party.',
     'Low', True, 'Hidden Crisis'),
]

results = []
for name, text, assigned, exp_mismatch, exp_type in ADVERSARIAL:
    rule_sev, rule_sc, _ = rule_score_single(text)
    ml, mt, delta = create_mismatch_label(assigned, rule_sev)
    got_mismatch = (ml == 'Mismatch')
    correct = (got_mismatch == exp_mismatch) and (not exp_mismatch or mt == exp_type)
    results.append({'Case':name,'Assigned':assigned,'Inferred':rule_sev,
                    'Expected':exp_type or 'Consistent','Got':mt or 'Consistent','Correct':correct})

adv_df = pd.DataFrame(results)
score  = adv_df['Correct'].sum()
print(f'Adversarial Score: {score}/{len(ADVERSARIAL)} = {score/len(ADVERSARIAL):.1%}')
print(f'{"✓ BONUS ELIGIBLE (≥7/10)" if score>=7 else "⚠ Below bonus threshold"}')
print()
for _,r in adv_df.iterrows():
    icon = '✓' if r['Correct'] else '✗'
    print(f'  {icon} {r["Case"][:40]:40s} | {r["Assigned"]:8s} → {r["Inferred"]:8s} | {r["Got"]}')

## 📊 Stage 9: Mismatch Analytics Dashboard

In [ ]:
fig = plt.figure(figsize=(18,12))
gs  = fig.add_gridspec(2,3, hspace=0.35, wspace=0.3)
PALETTE = {'Low':'#28a745','Medium':'#ffc107','High':'#fd7e14','Critical':'#dc3545'}

# 1. Mismatch by assigned priority
ax1 = fig.add_subplot(gs[0,0])
mm_rate = df.groupby(COL_PRIORITY)['label'].mean().reindex(PRIORITY_LEVELS)
ax1.bar(mm_rate.index, mm_rate.values*100, color=[PALETTE[p] for p in mm_rate.index], edgecolor='white')
ax1.axhline(50, color='red', linestyle='--', alpha=0.5)
ax1.set_title('Mismatch Rate by Assigned Priority', fontweight='bold')
ax1.set_ylabel('Mismatch Rate (%)'); ax1.set_ylim(0,110)
for i,(k,v) in enumerate(mm_rate.items()): ax1.text(i,v*100+2,f'{v*100:.0f}%',ha='center',fontweight='bold')

# 2. Mismatch type pie
ax2 = fig.add_subplot(gs[0,1])
type_counts = df['mismatch_type'].replace('','Consistent').value_counts()
colors_pie  = {'Hidden Crisis':'#dc3545','False Alarm':'#fd7e14','Consistent':'#28a745'}
ax2.pie(type_counts.values, labels=type_counts.index,
        colors=[colors_pie.get(k,'#999') for k in type_counts.index],
        autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax2.set_title('Mismatch Type Distribution', fontweight='bold')

# 3. Inferred severity distribution
ax3 = fig.add_subplot(gs[0,2])
sev_counts = df['inferred_severity'].value_counts().reindex(PRIORITY_LEVELS)
ax3.bar(sev_counts.index, sev_counts.values, color=[PALETTE[p] for p in sev_counts.index], edgecolor='white')
ax3.set_title('Inferred Severity Distribution', fontweight='bold'); ax3.set_ylabel('Count')

# 4. Severity delta heatmap
ax4 = fig.add_subplot(gs[1,0])
pivot = pd.crosstab(df[COL_PRIORITY], df['inferred_severity'])
pivot = pivot.reindex(index=PRIORITY_LEVELS, columns=PRIORITY_LEVELS, fill_value=0)
sns.heatmap(pivot, annot=True, fmt='d', cmap='RdBu_r', center=0, ax=ax4, linewidths=0.5)
ax4.set_title('Assigned vs Inferred (Heatmap)', fontweight='bold')

# 5. Mismatches by channel
ax5 = fig.add_subplot(gs[1,1])
ch_mm = df.groupby(COL_CHANNEL)['label'].agg(['sum','count'])
ch_mm['rate'] = ch_mm['sum']/ch_mm['count']*100
ax5.bar(ch_mm.index, ch_mm['rate'], color='#9b59b6', edgecolor='white')
ax5.set_title('Mismatch Rate by Channel', fontweight='bold'); ax5.set_ylabel('%')
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=30, ha='right')

# 6. Signal agreement distribution
ax6 = fig.add_subplot(gs[1,2])
ax6.hist(df['signal_agreement'], bins=10, color='#1abc9c', edgecolor='white')
ax6.axvline(df['signal_agreement'].mean(), color='red', linestyle='--',
            label=f'Mean={df["signal_agreement"].mean():.2f}')
ax6.set_title('Signal Agreement Distribution', fontweight='bold')
ax6.set_xlabel('Agreement (0–1)'); ax6.legend()

fig.suptitle('SIA — Priority Mismatch Analytics Dashboard', fontsize=16, fontweight='bold')
plt.show()

## ✅ Pipeline Summary

In [ ]:
print('=' * 60)
print('  SUPPORT INTEGRITY AUDITOR — PIPELINE SUMMARY')
print('=' * 60)
print(f'  Dataset:          {len(df)} tickets')
print(f'  Mismatches:       {df["label"].sum()} ({df["label"].mean()*100:.1f}%)')
print(f'  Hidden Crisis:    {(df["mismatch_type"]=="Hidden Crisis").sum()}')
print(f'  False Alarm:      {(df["mismatch_type"]=="False Alarm").sum()}')
print(f'  Signal Agreement: {df["signal_agreement"].mean():.3f} avg')
print()
print(f'  [Baseline TF-IDF+LR]')
print(f'  Accuracy:         {acc*100:.2f}%  {"✓" if acc>=0.83 else "✗"}')
print(f'  Macro F1:         {macro_f1:.4f}  {"✓" if macro_f1>=0.82 else "✗"}')
print(f'  Consistent Recall:{per_rec[0]:.4f}  {"✓" if per_rec[0]>=0.78 else "✗"}')
print(f'  Mismatch Recall:  {per_rec[1]:.4f}  {"✓" if per_rec[1]>=0.78 else "✗"}')
print(f'  ROC-AUC:          {roc:.4f}')
print()
print(f'  Dossiers:         {len(dossiers)} (zero hallucination)')
print(f'  Adversarial:      {score}/{len(ADVERSARIAL)} = {score/len(ADVERSARIAL):.1%}')
print()
overall = acc>=0.83 and macro_f1>=0.82 and per_rec[0]>=0.78 and per_rec[1]>=0.78
print(f'  MARS 2026 Status: {"✓ ALL THRESHOLDS MET — SUBMISSION READY" if overall else "⚠ REVIEW NEEDED"}')
print('=' * 60)